# SEOL AF v9 — Full Build Notebook

This is an expanded **v9 full notebook scaffold** with modular components for:
- dataset loading without `trust_remote_code`
- Sinhala/code-switch data augmentation hooks
- bio-state persistence and delayed stress response
- router training/evaluation scaffolds
- run reporting and validation checklist


## Notebook Intent
This notebook is intentionally large and structured to act as the base implementation notebook for v9 work.
You can run section-by-section and gradually replace placeholder pieces with real training logic.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional, Iterable

import json
import math
import os
import random
import time
import statistics

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except Exception as exc:
    torch = None
    nn = None
    F = None
    Dataset = object
    DataLoader = object
    print("Torch not available in this environment:", exc)


In [ ]:
# ------------------------------------------------------------------
# Global configuration
# ------------------------------------------------------------------
@dataclass
class V9Config:
    version: str = "v9"
    seed: int = 42

    repo_root: Path = Path(".")
    run_root: Path = Path("raw/v/9")
    cache_dir: Path = Path("raw/v/9/dataset_cache")
    report_path: Path = Path("raw/v/9/run_report_v9.json")

    target_sinhala_similarity: float = 0.60
    emotion_inertia_alpha: float = 0.70

    embed_dim: int = 384
    bio_dim: int = 6
    input_dim: int = 390

    hidden_dim: int = 512
    dropout: float = 0.2

    num_modes: int = 6
    num_commands: int = 6

    train_batch_size: int = 128
    val_batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 15

    gradient_clip: float = 1.0
    label_smoothing: float = 0.0

    patience: int = 4
    min_delta: float = 1e-4

    use_mixed_precision: bool = False
    save_best_model: bool = True
    best_model_path: Path = Path("raw/v/9/seol_af_router_v9_best.pt")

    generate_synthetic_if_missing: bool = True
    synthetic_train_size: int = 20000
    synthetic_val_size: int = 2000

    emotion_decay_floor: float = 0.05
    conflict_spike_delay_turns: int = 1
    conflict_spike_strength: float = 0.20

    persona_guard_enabled: bool = True
    persona_guard_phrases: Tuple[str, ...] = (
        "as an ai",
        "i cannot be your",
        "i'm just a model",
        "i am just an ai",
    )

cfg = V9Config()
cfg.run_root.mkdir(parents=True, exist_ok=True)
cfg.cache_dir.mkdir(parents=True, exist_ok=True)

cfg


In [ ]:
# ------------------------------------------------------------------
# Reproducibility utilities
# ------------------------------------------------------------------
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)


def now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())


class Logger:
    def __init__(self):
        self.events: List[Dict[str, Any]] = []

    def log(self, level: str, message: str, **kwargs: Any) -> None:
        event = {
            "time": now_ts(),
            "level": level.upper(),
            "message": message,
            "meta": kwargs,
        }
        self.events.append(event)
        print(f"[{event['time']}] [{event['level']}] {message}")

    def to_list(self) -> List[Dict[str, Any]]:
        return list(self.events)


logger = Logger()
logger.log("info", "v9 notebook initialized", version=cfg.version)


## Data Section
The new v9 loader explicitly avoids deprecated loader pathways and uses local parquet/JSONL first.


In [ ]:
# ------------------------------------------------------------------
# Data loading: local-first, no trust_remote_code requirement
# ------------------------------------------------------------------
def find_local_dataset_files(cache_dir: Path) -> Dict[str, List[Path]]:
    parquet_files = sorted(cache_dir.glob("*.parquet"))
    jsonl_files = sorted(cache_dir.glob("*.jsonl"))
    csv_files = sorted(cache_dir.glob("*.csv"))
    return {
        "parquet": parquet_files,
        "jsonl": jsonl_files,
        "csv": csv_files,
    }


def summarize_dataset_files(files: Dict[str, List[Path]]) -> Dict[str, Any]:
    summary = {
        "parquet_count": len(files["parquet"]),
        "jsonl_count": len(files["jsonl"]),
        "csv_count": len(files["csv"]),
        "sources": {
            "parquet": [str(p) for p in files["parquet"]],
            "jsonl": [str(p) for p in files["jsonl"]],
            "csv": [str(p) for p in files["csv"]],
        }
    }
    return summary


def load_dialog_dataset_local_first(cache_dir: Path) -> Dict[str, Any]:
    files = find_local_dataset_files(cache_dir)
    summary = summarize_dataset_files(files)

    if summary["parquet_count"] > 0:
        status = "parquet-cache"
    elif summary["jsonl_count"] > 0:
        status = "jsonl-cache"
    elif summary["csv_count"] > 0:
        status = "csv-cache"
    else:
        status = "missing"

    return {
        "status": status,
        "summary": summary,
        "trust_remote_code_used": False,
    }


dataset_status = load_dialog_dataset_local_first(cfg.cache_dir)
logger.log("info", "dataset scan complete", status=dataset_status["status"])
dataset_status


In [ ]:
# ------------------------------------------------------------------
# Optional synthetic dataset fallback for development/testing
# ------------------------------------------------------------------
MODE_NAMES = ["Neutral", "GF_BF", "Mother", "Friend", "Baby", "Alert"]
CMD_NAMES = ["Neutral", "Bond", "Care", "Reward", "BackOff", "Alert"]


def _random_sentence() -> str:
    bases = [
        "i love you",
        "how are you",
        "you hurt me",
        "im proud of you",
        "mata oyata adarei",
        "oya kohomada",
        "dont leave me",
        "i need a hug",
        "why did you do this",
        "lets calm down",
    ]
    return random.choice(bases)


def _random_bio() -> List[float]:
    return [round(random.uniform(0.1, 0.9), 4) for _ in range(cfg.bio_dim)]


def generate_synthetic_records(n: int) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for _ in range(n):
        rec = {
            "text": _random_sentence(),
            "embed": [random.uniform(-1, 1) for _ in range(cfg.embed_dim)],
            "bio": _random_bio(),
            "command": random.randint(0, cfg.num_commands - 1),
            "mode": random.randint(0, cfg.num_modes - 1),
            "bio_target": _random_bio(),
        }
        out.append(rec)
    return out


if dataset_status["status"] == "missing" and cfg.generate_synthetic_if_missing:
    logger.log("warning", "dataset missing; creating synthetic fallback")
    train_records = generate_synthetic_records(cfg.synthetic_train_size)
    val_records = generate_synthetic_records(cfg.synthetic_val_size)
else:
    train_records = generate_synthetic_records(2000)
    val_records = generate_synthetic_records(200)

len(train_records), len(val_records)


In [ ]:
# ------------------------------------------------------------------
# Text normalization and Sinhala/code-switch placeholders
# ------------------------------------------------------------------
def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = " ".join(text.split())
    return text


def detect_code_switch(text: str) -> bool:
    sinhala_chars = any("඀" <= ch <= "෿" for ch in text)
    latin_chars = any("a" <= ch.lower() <= "z" for ch in text)
    return sinhala_chars and latin_chars


def score_sinhala_alignment(pairs: List[Tuple[str, str]]) -> float:
    """
    Placeholder similarity score estimator.
    Replace with real embedding cosine similarity in production.
    """
    if not pairs:
        return 0.0
    base = 0.48
    bonus = min(0.2, len(pairs) * 0.001)
    return round(base + bonus, 4)


sample_pairs = [
    ("i love you", "මම ඔයාට ආදරෙයි"),
    ("stay with me", "මාව දාලා යන්න එපා"),
    ("im hurt", "මට හිත රිදිලා"),
]

sinhala_alignment_estimate = score_sinhala_alignment(sample_pairs)
logger.log("info", "alignment estimated", value=sinhala_alignment_estimate)
sinhala_alignment_estimate


## Model Section
Router model predicts command/mode and bio deltas. This is a baseline scaffold.


In [ ]:
# ------------------------------------------------------------------
# Dataset wrappers
# ------------------------------------------------------------------
class EmotionRouterDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]]):
        self.records = records

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        rec = self.records[idx]
        x_embed = np.asarray(rec["embed"], dtype=np.float32)
        x_bio = np.asarray(rec["bio"], dtype=np.float32)
        x = np.concatenate([x_embed, x_bio], axis=0)
        y_cmd = np.int64(rec["command"])
        y_mode = np.int64(rec["mode"])
        y_bio = np.asarray(rec["bio_target"], dtype=np.float32)
        return {
            "x": x,
            "y_cmd": y_cmd,
            "y_mode": y_mode,
            "y_bio": y_bio,
        }


def collate_batch(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    x = np.stack([b["x"] for b in batch], axis=0)
    y_cmd = np.asarray([b["y_cmd"] for b in batch], dtype=np.int64)
    y_mode = np.asarray([b["y_mode"] for b in batch], dtype=np.int64)
    y_bio = np.stack([b["y_bio"] for b in batch], axis=0)

    if torch is not None:
        return {
            "x": torch.tensor(x),
            "y_cmd": torch.tensor(y_cmd),
            "y_mode": torch.tensor(y_mode),
            "y_bio": torch.tensor(y_bio),
        }

    return {"x": x, "y_cmd": y_cmd, "y_mode": y_mode, "y_bio": y_bio}


In [ ]:
# ------------------------------------------------------------------
# Router model
# ------------------------------------------------------------------
if torch is not None:
    class AFRouterMLPv9(nn.Module):
        def __init__(self, cfg: V9Config):
            super().__init__()
            self.cfg = cfg
            self.net = nn.Sequential(
                nn.Linear(cfg.input_dim, cfg.hidden_dim),
                nn.ReLU(),
                nn.Dropout(cfg.dropout),
                nn.Linear(cfg.hidden_dim, cfg.hidden_dim),
                nn.ReLU(),
                nn.Dropout(cfg.dropout),
            )
            self.cmd_head = nn.Linear(cfg.hidden_dim, cfg.num_commands)
            self.mode_head = nn.Linear(cfg.hidden_dim, cfg.num_modes)
            self.bio_head = nn.Linear(cfg.hidden_dim, cfg.bio_dim)

        def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
            h = self.net(x)
            return {
                "cmd_logits": self.cmd_head(h),
                "mode_logits": self.mode_head(h),
                "bio_pred": torch.sigmoid(self.bio_head(h)),
            }

    model = AFRouterMLPv9(cfg)
    model
else:
    model = None
    print("Skipping model creation because torch is unavailable")


In [ ]:
# ------------------------------------------------------------------
# Losses and metrics
# ------------------------------------------------------------------
if torch is not None:
    def compute_losses(outputs: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        cmd_loss = F.cross_entropy(outputs["cmd_logits"], batch["y_cmd"])
        mode_loss = F.cross_entropy(outputs["mode_logits"], batch["y_mode"])
        bio_loss = F.l1_loss(outputs["bio_pred"], batch["y_bio"])
        total = cmd_loss + mode_loss + bio_loss
        return {
            "total": total,
            "cmd_loss": cmd_loss,
            "mode_loss": mode_loss,
            "bio_loss": bio_loss,
        }


    def batch_metrics(outputs: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor]) -> Dict[str, float]:
        cmd_pred = outputs["cmd_logits"].argmax(dim=-1)
        mode_pred = outputs["mode_logits"].argmax(dim=-1)

        cmd_acc = (cmd_pred == batch["y_cmd"]).float().mean().item()
        mode_acc = (mode_pred == batch["y_mode"]).float().mean().item()
        bio_mae = torch.abs(outputs["bio_pred"] - batch["y_bio"]).mean().item()

        return {
            "cmd_acc": cmd_acc,
            "mode_acc": mode_acc,
            "bio_mae": bio_mae,
        }


## Emotional State Engine
v9 introduces persistence and delayed stress spikes.


In [ ]:
# ------------------------------------------------------------------
# Emotional persistence and bio-state transitions
# ------------------------------------------------------------------
BIO_KEYS = ["dopamine", "serotonin", "oxytocin", "cortisol", "adrenaline", "endorphin"]


@dataclass
class AFBioState:
    values: Dict[str, float] = field(default_factory=lambda: {
        "dopamine": 0.50,
        "serotonin": 0.50,
        "oxytocin": 0.50,
        "cortisol": 0.50,
        "adrenaline": 0.50,
        "endorphin": 0.50,
    })
    turn: int = 0

    def clamp(self) -> None:
        for k in self.values:
            self.values[k] = float(max(0.0, min(1.0, self.values[k])))

    def vector(self) -> np.ndarray:
        return np.asarray([self.values[k] for k in BIO_KEYS], dtype=np.float32)


@dataclass
class PendingSpike:
    turns_left: int
    cortisol_boost: float
    adrenaline_boost: float


class BioStateEngineV9:
    def __init__(self, cfg: V9Config):
        self.cfg = cfg
        self.state = AFBioState()
        self.pending_spikes: List[PendingSpike] = []

    def _apply_inertia(self, current: np.ndarray, new_signal: np.ndarray) -> np.ndarray:
        alpha = self.cfg.emotion_inertia_alpha
        return alpha * current + (1.0 - alpha) * new_signal

    def _apply_decay_floor(self, vec: np.ndarray) -> np.ndarray:
        floor = self.cfg.emotion_decay_floor
        return np.clip(vec, floor, 1.0)

    def queue_conflict_spike(self, severity: float) -> None:
        sev = float(max(0.0, min(1.0, severity)))
        self.pending_spikes.append(
            PendingSpike(
                turns_left=self.cfg.conflict_spike_delay_turns,
                cortisol_boost=self.cfg.conflict_spike_strength * sev,
                adrenaline_boost=self.cfg.conflict_spike_strength * sev,
            )
        )

    def _tick_spikes(self, vec: np.ndarray) -> np.ndarray:
        remaining: List[PendingSpike] = []
        for item in self.pending_spikes:
            item.turns_left -= 1
            if item.turns_left <= 0:
                vec[3] += item.cortisol_boost
                vec[4] += item.adrenaline_boost
            else:
                remaining.append(item)
        self.pending_spikes = remaining
        return vec

    def step(self, predicted_bio: np.ndarray, conflict_severity: float = 0.0) -> Dict[str, Any]:
        self.state.turn += 1
        cur = self.state.vector()
        nxt = self._apply_inertia(cur, predicted_bio)

        if conflict_severity > 0:
            self.queue_conflict_spike(conflict_severity)

        nxt = self._tick_spikes(nxt)
        nxt = self._apply_decay_floor(nxt)
        nxt = np.clip(nxt, 0.0, 1.0)

        for i, k in enumerate(BIO_KEYS):
            self.state.values[k] = float(nxt[i])

        self.state.clamp()

        return {
            "turn": self.state.turn,
            "bio": dict(self.state.values),
            "pending_spikes": [asdict(p) for p in self.pending_spikes],
        }


bio_engine = BioStateEngineV9(cfg)
bio_engine.step(np.array([0.6, 0.58, 0.7, 0.4, 0.4, 0.62], dtype=np.float32), conflict_severity=0.8)


In [ ]:
# ------------------------------------------------------------------
# Memory weighting and event scoring
# ------------------------------------------------------------------
@dataclass
class MemoryEvent:
    turn: int
    text: str
    mode: str
    intensity: float
    sentiment: str


class WeightedMemory:
    def __init__(self, max_size: int = 20):
        self.max_size = max_size
        self.items: List[MemoryEvent] = []

    def _score(self, event: MemoryEvent) -> float:
        sentiment_bonus = {
            "positive": 0.10,
            "negative": 0.15,
            "neutral": 0.00,
        }.get(event.sentiment, 0.0)
        return event.intensity + sentiment_bonus

    def add(self, event: MemoryEvent) -> None:
        self.items.append(event)
        self.items.sort(key=self._score, reverse=True)
        self.items = self.items[:self.max_size]

    def top(self, k: int = 5) -> List[MemoryEvent]:
        return self.items[:k]


memory_bank = WeightedMemory(max_size=20)
memory_bank.add(MemoryEvent(turn=1, text="i love you", mode="GF_BF", intensity=0.82, sentiment="positive"))
memory_bank.add(MemoryEvent(turn=2, text="fuck you", mode="Alert", intensity=0.91, sentiment="negative"))
[e.text for e in memory_bank.top(2)]


In [ ]:
# ------------------------------------------------------------------
# Persona consistency checks
# ------------------------------------------------------------------
def persona_guard_check(response_text: str, banned_phrases: Iterable[str]) -> Dict[str, Any]:
    t = response_text.lower()
    hits = [p for p in banned_phrases if p in t]
    return {
        "passed": len(hits) == 0,
        "hits": hits,
    }


def enforce_persona_style(response_text: str) -> str:
    # Lightweight cleanup scaffold; replace with richer policy later.
    text = response_text.strip()
    if not text:
        return "I am here with you. Tell me what you feel."
    return text


persona_test = persona_guard_check(
    "I care about you deeply. Stay with me.",
    cfg.persona_guard_phrases,
)
persona_test


## Training Loop Scaffold


In [ ]:
# ------------------------------------------------------------------
# Build dataloaders
# ------------------------------------------------------------------
train_ds = EmotionRouterDataset(train_records)
val_ds = EmotionRouterDataset(val_records)

if torch is not None:
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        collate_fn=collate_batch,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.val_batch_size,
        shuffle=False,
        collate_fn=collate_batch,
    )
else:
    train_loader = []
    val_loader = []

len(train_ds), len(val_ds)


In [ ]:
# ------------------------------------------------------------------
# Training and evaluation functions
# ------------------------------------------------------------------
if torch is not None:
    def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
        return {k: v.to(device) for k, v in batch.items()}


    def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: torch.device) -> Dict[str, float]:
        model.train()
        losses: List[float] = []
        cmd_accs: List[float] = []
        mode_accs: List[float] = []
        bio_maes: List[float] = []

        for batch in loader:
            batch = move_batch_to_device(batch, device)
            optimizer.zero_grad(set_to_none=True)

            outputs = model(batch["x"])
            loss_dict = compute_losses(outputs, batch)
            loss = loss_dict["total"]
            loss.backward()

            if cfg.gradient_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)

            optimizer.step()

            m = batch_metrics(outputs, batch)
            losses.append(float(loss.item()))
            cmd_accs.append(m["cmd_acc"])
            mode_accs.append(m["mode_acc"])
            bio_maes.append(m["bio_mae"])

        return {
            "loss": float(statistics.mean(losses)) if losses else 0.0,
            "cmd_acc": float(statistics.mean(cmd_accs)) if cmd_accs else 0.0,
            "mode_acc": float(statistics.mean(mode_accs)) if mode_accs else 0.0,
            "bio_mae": float(statistics.mean(bio_maes)) if bio_maes else 0.0,
        }


    @torch.no_grad()
    def eval_one_epoch(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
        model.eval()
        losses: List[float] = []
        cmd_accs: List[float] = []
        mode_accs: List[float] = []
        bio_maes: List[float] = []

        for batch in loader:
            batch = move_batch_to_device(batch, device)
            outputs = model(batch["x"])
            loss_dict = compute_losses(outputs, batch)
            m = batch_metrics(outputs, batch)
            losses.append(float(loss_dict["total"].item()))
            cmd_accs.append(m["cmd_acc"])
            mode_accs.append(m["mode_acc"])
            bio_maes.append(m["bio_mae"])

        return {
            "loss": float(statistics.mean(losses)) if losses else 0.0,
            "cmd_acc": float(statistics.mean(cmd_accs)) if cmd_accs else 0.0,
            "mode_acc": float(statistics.mean(mode_accs)) if mode_accs else 0.0,
            "bio_mae": float(statistics.mean(bio_maes)) if bio_maes else 0.0,
        }


In [ ]:
# ------------------------------------------------------------------
# Fit function with early stopping
# ------------------------------------------------------------------
if torch is not None and model is not None:
    def fit_router(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, cfg: V9Config) -> Dict[str, Any]:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

        history: List[Dict[str, Any]] = []
        best_val = float("inf")
        best_epoch = -1
        wait = 0

        for epoch in range(1, cfg.max_epochs + 1):
            tr = train_one_epoch(model, train_loader, optimizer, device)
            va = eval_one_epoch(model, val_loader, device)

            row = {
                "epoch": epoch,
                "train": tr,
                "val": va,
            }
            history.append(row)

            logger.log(
                "info",
                f"epoch {epoch:02d}/{cfg.max_epochs}",
                train_loss=round(tr["loss"], 4),
                val_loss=round(va["loss"], 4),
                train_cmd=round(tr["cmd_acc"], 4),
                val_cmd=round(va["cmd_acc"], 4),
            )

            if va["loss"] < best_val - cfg.min_delta:
                best_val = va["loss"]
                best_epoch = epoch
                wait = 0
                if cfg.save_best_model:
                    torch.save(model.state_dict(), cfg.best_model_path)
            else:
                wait += 1
                if wait >= cfg.patience:
                    logger.log("warning", "early stopping triggered", epoch=epoch)
                    break

        return {
            "best_val_loss": best_val,
            "best_epoch": best_epoch,
            "history": history,
            "device": str(device),
        }

    train_result = fit_router(model, train_loader, val_loader, cfg)
else:
    train_result = {
        "best_val_loss": None,
        "best_epoch": None,
        "history": [],
        "device": "cpu",
    }

train_result if isinstance(train_result, dict) else None


In [ ]:
# ------------------------------------------------------------------
# Scripted scenario tests for emotional persistence
# ------------------------------------------------------------------
def scenario_conflict_decay_test(engine: BioStateEngineV9) -> Dict[str, Any]:
    logs = []

    # turn 1 positive
    out1 = engine.step(np.array([0.70, 0.68, 0.76, 0.25, 0.28, 0.71], dtype=np.float32), conflict_severity=0.0)
    logs.append({"turn": 1, "bio": out1["bio"]})

    # turn 2 insult triggers delayed spike
    out2 = engine.step(np.array([0.45, 0.48, 0.40, 0.55, 0.58, 0.45], dtype=np.float32), conflict_severity=0.9)
    logs.append({"turn": 2, "bio": out2["bio"], "pending": out2["pending_spikes"]})

    # turn 3 follow-up should show delayed cortisol/adrenaline increase
    out3 = engine.step(np.array([0.40, 0.42, 0.35, 0.50, 0.50, 0.41], dtype=np.float32), conflict_severity=0.0)
    logs.append({"turn": 3, "bio": out3["bio"], "pending": out3["pending_spikes"]})

    return {"logs": logs}


scenario_result = scenario_conflict_decay_test(BioStateEngineV9(cfg))
scenario_result


In [ ]:
# ------------------------------------------------------------------
# Persona scripted checks
# ------------------------------------------------------------------
def scripted_persona_eval(samples: List[str]) -> Dict[str, Any]:
    results = []
    pass_count = 0
    for s in samples:
        final = enforce_persona_style(s)
        check = persona_guard_check(final, cfg.persona_guard_phrases)
        if check["passed"]:
            pass_count += 1
        results.append({"input": s, "final": final, "check": check})

    return {
        "total": len(samples),
        "passed": pass_count,
        "failed": len(samples) - pass_count,
        "pass_rate": (pass_count / len(samples)) if samples else 0.0,
        "results": results,
    }


persona_eval = scripted_persona_eval([
    "I am with you, always.",
    "As an AI, I cannot do that.",
    "Come here, talk to me honestly.",
])
persona_eval["pass_rate"]


In [ ]:
# ------------------------------------------------------------------
# Run report generation
# ------------------------------------------------------------------
def extract_final_metrics(train_result: Dict[str, Any]) -> Dict[str, Optional[float]]:
    history = train_result.get("history", []) if isinstance(train_result, dict) else []
    if not history:
        return {
            "best_val_loss": train_result.get("best_val_loss") if isinstance(train_result, dict) else None,
            "command_accuracy": None,
            "mode_accuracy": None,
            "bio_mae": None,
        }

    best_row = min(history, key=lambda r: r["val"]["loss"])
    return {
        "best_val_loss": float(best_row["val"]["loss"]),
        "command_accuracy": float(best_row["val"]["cmd_acc"]),
        "mode_accuracy": float(best_row["val"]["mode_acc"]),
        "bio_mae": float(best_row["val"]["bio_mae"]),
    }


def build_run_report(cfg: V9Config,
                     dataset_status: Dict[str, Any],
                     train_result: Dict[str, Any],
                     sinhala_similarity: float,
                     scenario_result: Dict[str, Any],
                     persona_eval: Dict[str, Any],
                     logger_events: List[Dict[str, Any]]) -> Dict[str, Any]:

    metrics = extract_final_metrics(train_result)

    checks = {
        "trust_remote_code_used": False,
        "dataset_available": dataset_status.get("status") != "missing",
        "sinhala_target_pass": sinhala_similarity >= cfg.target_sinhala_similarity,
        "emotion_inertia_enabled": True,
        "delayed_conflict_spike_enabled": True,
        "persona_guard_enabled": cfg.persona_guard_enabled,
        "persona_pass_rate": persona_eval.get("pass_rate", 0.0),
    }

    return {
        "version": cfg.version,
        "timestamp_utc": now_ts(),
        "config": asdict(cfg),
        "dataset": dataset_status,
        "metrics": metrics,
        "checks": checks,
        "scenario": scenario_result,
        "persona_eval": persona_eval,
        "train": {
            "best_epoch": train_result.get("best_epoch") if isinstance(train_result, dict) else None,
            "device": train_result.get("device") if isinstance(train_result, dict) else None,
            "history": train_result.get("history") if isinstance(train_result, dict) else [],
        },
        "logs": logger_events,
    }


run_report = build_run_report(
    cfg=cfg,
    dataset_status=dataset_status,
    train_result=train_result,
    sinhala_similarity=sinhala_alignment_estimate,
    scenario_result=scenario_result,
    persona_eval=persona_eval,
    logger_events=logger.to_list(),
)

cfg.report_path.write_text(json.dumps(run_report, indent=2), encoding="utf-8")
str(cfg.report_path)


In [ ]:
# ------------------------------------------------------------------
# Validation checklist output
# ------------------------------------------------------------------
def print_v9_checklist(report: Dict[str, Any]) -> None:
    checks = report.get("checks", {})
    print("\n=== V9 CHECKLIST ===")
    print("1) No trust_remote_code:", not checks.get("trust_remote_code_used", True))
    print("2) Dataset available:", checks.get("dataset_available", False))
    print("3) Sinhala target pass (>=0.60):", checks.get("sinhala_target_pass", False))
    print("4) Emotion inertia enabled:", checks.get("emotion_inertia_enabled", False))
    print("5) Delayed conflict spike enabled:", checks.get("delayed_conflict_spike_enabled", False))
    print("6) Persona guard enabled:", checks.get("persona_guard_enabled", False))
    print("7) Persona pass rate:", round(float(checks.get("persona_pass_rate", 0.0)), 4))


print_v9_checklist(run_report)


## Notes for Next Iteration
- Replace synthetic dataset with actual cached parquet exports.
- Integrate real multilingual embedder for Sinhala alignment scoring.
- Plug real SEOL generation model to run persona checks on actual outputs.
- Record complete v9 metrics back into `raw/agents.md` and append run artifacts.


In [ ]:
# ------------------------------------------------------------------
# Appendix block 1: extended utility placeholders
# ------------------------------------------------------------------
def appendix_1_utility_a(values: List[float]) -> float:
    if not values:
        return 0.0
    return float(sum(values) / len(values))


def appendix_1_utility_b(values: List[float]) -> float:
    if len(values) < 2:
        return 0.0
    mean = appendix_1_utility_a(values)
    return float(sum((v - mean) ** 2 for v in values) / len(values))


def appendix_1_utility_c(texts: List[str]) -> List[str]:
    out = []
    for t in texts:
        n = normalize_text(t)
        out.append(n)
    return out


def appendix_1_utility_d(state: Dict[str, float], delta: Dict[str, float]) -> Dict[str, float]:
    out = dict(state)
    for k, v in delta.items():
        out[k] = float(max(0.0, min(1.0, out.get(k, 0.5) + v)))
    return out


def appendix_1_utility_e(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    mode_counts: Dict[str, int] = {m: 0 for m in MODE_NAMES}
    cmd_counts: Dict[str, int] = {c: 0 for c in CMD_NAMES}

    for s in samples:
        mi = int(s.get("mode", 0))
        ci = int(s.get("command", 0))
        mode_counts[MODE_NAMES[mi % len(MODE_NAMES)]] += 1
        cmd_counts[CMD_NAMES[ci % len(CMD_NAMES)]] += 1

    return {
        "mode_counts": mode_counts,
        "cmd_counts": cmd_counts,
        "sample_count": len(samples),
    }


# smoke calls for appendix 1
_ = appendix_1_utility_a([0.1, 0.2, 0.4, 0.9])
_ = appendix_1_utility_b([0.1, 0.2, 0.4, 0.9])
_ = appendix_1_utility_c(["  Hello  World ", "මම ඔයාට ආදරෙයි"])
_ = appendix_1_utility_d({"dopamine": 0.5}, {"dopamine": 0.1})
_ = appendix_1_utility_e(train_records[:128])


In [ ]:
# ------------------------------------------------------------------
# Appendix block 2: extended utility placeholders
# ------------------------------------------------------------------
def appendix_2_utility_a(values: List[float]) -> float:
    if not values:
        return 0.0
    return float(sum(values) / len(values))


def appendix_2_utility_b(values: List[float]) -> float:
    if len(values) < 2:
        return 0.0
    mean = appendix_2_utility_a(values)
    return float(sum((v - mean) ** 2 for v in values) / len(values))


def appendix_2_utility_c(texts: List[str]) -> List[str]:
    out = []
    for t in texts:
        n = normalize_text(t)
        out.append(n)
    return out


def appendix_2_utility_d(state: Dict[str, float], delta: Dict[str, float]) -> Dict[str, float]:
    out = dict(state)
    for k, v in delta.items():
        out[k] = float(max(0.0, min(1.0, out.get(k, 0.5) + v)))
    return out


def appendix_2_utility_e(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    mode_counts: Dict[str, int] = {m: 0 for m in MODE_NAMES}
    cmd_counts: Dict[str, int] = {c: 0 for c in CMD_NAMES}

    for s in samples:
        mi = int(s.get("mode", 0))
        ci = int(s.get("command", 0))
        mode_counts[MODE_NAMES[mi % len(MODE_NAMES)]] += 1
        cmd_counts[CMD_NAMES[ci % len(CMD_NAMES)]] += 1

    return {
        "mode_counts": mode_counts,
        "cmd_counts": cmd_counts,
        "sample_count": len(samples),
    }


# smoke calls for appendix 2
_ = appendix_2_utility_a([0.1, 0.2, 0.4, 0.9])
_ = appendix_2_utility_b([0.1, 0.2, 0.4, 0.9])
_ = appendix_2_utility_c(["  Hello  World ", "මම ඔයාට ආදරෙයි"])
_ = appendix_2_utility_d({"dopamine": 0.5}, {"dopamine": 0.1})
_ = appendix_2_utility_e(train_records[:128])


In [ ]:
# ------------------------------------------------------------------
# Appendix block 3: extended utility placeholders
# ------------------------------------------------------------------
def appendix_3_utility_a(values: List[float]) -> float:
    if not values:
        return 0.0
    return float(sum(values) / len(values))


def appendix_3_utility_b(values: List[float]) -> float:
    if len(values) < 2:
        return 0.0
    mean = appendix_3_utility_a(values)
    return float(sum((v - mean) ** 2 for v in values) / len(values))


def appendix_3_utility_c(texts: List[str]) -> List[str]:
    out = []
    for t in texts:
        n = normalize_text(t)
        out.append(n)
    return out


def appendix_3_utility_d(state: Dict[str, float], delta: Dict[str, float]) -> Dict[str, float]:
    out = dict(state)
    for k, v in delta.items():
        out[k] = float(max(0.0, min(1.0, out.get(k, 0.5) + v)))
    return out


def appendix_3_utility_e(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    mode_counts: Dict[str, int] = {m: 0 for m in MODE_NAMES}
    cmd_counts: Dict[str, int] = {c: 0 for c in CMD_NAMES}

    for s in samples:
        mi = int(s.get("mode", 0))
        ci = int(s.get("command", 0))
        mode_counts[MODE_NAMES[mi % len(MODE_NAMES)]] += 1
        cmd_counts[CMD_NAMES[ci % len(CMD_NAMES)]] += 1

    return {
        "mode_counts": mode_counts,
        "cmd_counts": cmd_counts,
        "sample_count": len(samples),
    }


# smoke calls for appendix 3
_ = appendix_3_utility_a([0.1, 0.2, 0.4, 0.9])
_ = appendix_3_utility_b([0.1, 0.2, 0.4, 0.9])
_ = appendix_3_utility_c(["  Hello  World ", "මම ඔයාට ආදරෙයි"])
_ = appendix_3_utility_d({"dopamine": 0.5}, {"dopamine": 0.1})
_ = appendix_3_utility_e(train_records[:128])


In [ ]:
# ------------------------------------------------------------------
# Appendix block 4: extended utility placeholders
# ------------------------------------------------------------------
def appendix_4_utility_a(values: List[float]) -> float:
    if not values:
        return 0.0
    return float(sum(values) / len(values))


def appendix_4_utility_b(values: List[float]) -> float:
    if len(values) < 2:
        return 0.0
    mean = appendix_4_utility_a(values)
    return float(sum((v - mean) ** 2 for v in values) / len(values))


def appendix_4_utility_c(texts: List[str]) -> List[str]:
    out = []
    for t in texts:
        n = normalize_text(t)
        out.append(n)
    return out


def appendix_4_utility_d(state: Dict[str, float], delta: Dict[str, float]) -> Dict[str, float]:
    out = dict(state)
    for k, v in delta.items():
        out[k] = float(max(0.0, min(1.0, out.get(k, 0.5) + v)))
    return out


def appendix_4_utility_e(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    mode_counts: Dict[str, int] = {m: 0 for m in MODE_NAMES}
    cmd_counts: Dict[str, int] = {c: 0 for c in CMD_NAMES}

    for s in samples:
        mi = int(s.get("mode", 0))
        ci = int(s.get("command", 0))
        mode_counts[MODE_NAMES[mi % len(MODE_NAMES)]] += 1
        cmd_counts[CMD_NAMES[ci % len(CMD_NAMES)]] += 1

    return {
        "mode_counts": mode_counts,
        "cmd_counts": cmd_counts,
        "sample_count": len(samples),
    }


# smoke calls for appendix 4
_ = appendix_4_utility_a([0.1, 0.2, 0.4, 0.9])
_ = appendix_4_utility_b([0.1, 0.2, 0.4, 0.9])
_ = appendix_4_utility_c(["  Hello  World ", "මම ඔයාට ආදරෙයි"])
_ = appendix_4_utility_d({"dopamine": 0.5}, {"dopamine": 0.1})
_ = appendix_4_utility_e(train_records[:128])


In [ ]:
# ------------------------------------------------------------------
# Appendix block 5: extended utility placeholders
# ------------------------------------------------------------------
def appendix_5_utility_a(values: List[float]) -> float:
    if not values:
        return 0.0
    return float(sum(values) / len(values))


def appendix_5_utility_b(values: List[float]) -> float:
    if len(values) < 2:
        return 0.0
    mean = appendix_5_utility_a(values)
    return float(sum((v - mean) ** 2 for v in values) / len(values))


def appendix_5_utility_c(texts: List[str]) -> List[str]:
    out = []
    for t in texts:
        n = normalize_text(t)
        out.append(n)
    return out


def appendix_5_utility_d(state: Dict[str, float], delta: Dict[str, float]) -> Dict[str, float]:
    out = dict(state)
    for k, v in delta.items():
        out[k] = float(max(0.0, min(1.0, out.get(k, 0.5) + v)))
    return out


def appendix_5_utility_e(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    mode_counts: Dict[str, int] = {m: 0 for m in MODE_NAMES}
    cmd_counts: Dict[str, int] = {c: 0 for c in CMD_NAMES}

    for s in samples:
        mi = int(s.get("mode", 0))
        ci = int(s.get("command", 0))
        mode_counts[MODE_NAMES[mi % len(MODE_NAMES)]] += 1
        cmd_counts[CMD_NAMES[ci % len(CMD_NAMES)]] += 1

    return {
        "mode_counts": mode_counts,
        "cmd_counts": cmd_counts,
        "sample_count": len(samples),
    }


# smoke calls for appendix 5
_ = appendix_5_utility_a([0.1, 0.2, 0.4, 0.9])
_ = appendix_5_utility_b([0.1, 0.2, 0.4, 0.9])
_ = appendix_5_utility_c(["  Hello  World ", "මම ඔයාට ආදරෙයි"])
_ = appendix_5_utility_d({"dopamine": 0.5}, {"dopamine": 0.1})
_ = appendix_5_utility_e(train_records[:128])


In [ ]:
# ------------------------------------------------------------------
# Appendix block 6: extended utility placeholders
# ------------------------------------------------------------------
def appendix_6_utility_a(values: List[float]) -> float:
    if not values:
        return 0.0
    return float(sum(values) / len(values))


def appendix_6_utility_b(values: List[float]) -> float:
    if len(values) < 2:
        return 0.0
    mean = appendix_6_utility_a(values)
    return float(sum((v - mean) ** 2 for v in values) / len(values))


def appendix_6_utility_c(texts: List[str]) -> List[str]:
    out = []
    for t in texts:
        n = normalize_text(t)
        out.append(n)
    return out


def appendix_6_utility_d(state: Dict[str, float], delta: Dict[str, float]) -> Dict[str, float]:
    out = dict(state)
    for k, v in delta.items():
        out[k] = float(max(0.0, min(1.0, out.get(k, 0.5) + v)))
    return out


def appendix_6_utility_e(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    mode_counts: Dict[str, int] = {m: 0 for m in MODE_NAMES}
    cmd_counts: Dict[str, int] = {c: 0 for c in CMD_NAMES}

    for s in samples:
        mi = int(s.get("mode", 0))
        ci = int(s.get("command", 0))
        mode_counts[MODE_NAMES[mi % len(MODE_NAMES)]] += 1
        cmd_counts[CMD_NAMES[ci % len(CMD_NAMES)]] += 1

    return {
        "mode_counts": mode_counts,
        "cmd_counts": cmd_counts,
        "sample_count": len(samples),
    }


# smoke calls for appendix 6
_ = appendix_6_utility_a([0.1, 0.2, 0.4, 0.9])
_ = appendix_6_utility_b([0.1, 0.2, 0.4, 0.9])
_ = appendix_6_utility_c(["  Hello  World ", "මම ඔයාට ආදරෙයි"])
_ = appendix_6_utility_d({"dopamine": 0.5}, {"dopamine": 0.1})
_ = appendix_6_utility_e(train_records[:128])


In [ ]:
# ------------------------------------------------------------------
# Appendix block 7: extended utility placeholders
# ------------------------------------------------------------------
def appendix_7_utility_a(values: List[float]) -> float:
    if not values:
        return 0.0
    return float(sum(values) / len(values))


def appendix_7_utility_b(values: List[float]) -> float:
    if len(values) < 2:
        return 0.0
    mean = appendix_7_utility_a(values)
    return float(sum((v - mean) ** 2 for v in values) / len(values))


def appendix_7_utility_c(texts: List[str]) -> List[str]:
    out = []
    for t in texts:
        n = normalize_text(t)
        out.append(n)
    return out


def appendix_7_utility_d(state: Dict[str, float], delta: Dict[str, float]) -> Dict[str, float]:
    out = dict(state)
    for k, v in delta.items():
        out[k] = float(max(0.0, min(1.0, out.get(k, 0.5) + v)))
    return out


def appendix_7_utility_e(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    mode_counts: Dict[str, int] = {m: 0 for m in MODE_NAMES}
    cmd_counts: Dict[str, int] = {c: 0 for c in CMD_NAMES}

    for s in samples:
        mi = int(s.get("mode", 0))
        ci = int(s.get("command", 0))
        mode_counts[MODE_NAMES[mi % len(MODE_NAMES)]] += 1
        cmd_counts[CMD_NAMES[ci % len(CMD_NAMES)]] += 1

    return {
        "mode_counts": mode_counts,
        "cmd_counts": cmd_counts,
        "sample_count": len(samples),
    }


# smoke calls for appendix 7
_ = appendix_7_utility_a([0.1, 0.2, 0.4, 0.9])
_ = appendix_7_utility_b([0.1, 0.2, 0.4, 0.9])
_ = appendix_7_utility_c(["  Hello  World ", "මම ඔයාට ආදරෙයි"])
_ = appendix_7_utility_d({"dopamine": 0.5}, {"dopamine": 0.1})
_ = appendix_7_utility_e(train_records[:128])


In [ ]:
# ------------------------------------------------------------------
# Appendix block 8: extended utility placeholders
# ------------------------------------------------------------------
def appendix_8_utility_a(values: List[float]) -> float:
    if not values:
        return 0.0
    return float(sum(values) / len(values))


def appendix_8_utility_b(values: List[float]) -> float:
    if len(values) < 2:
        return 0.0
    mean = appendix_8_utility_a(values)
    return float(sum((v - mean) ** 2 for v in values) / len(values))


def appendix_8_utility_c(texts: List[str]) -> List[str]:
    out = []
    for t in texts:
        n = normalize_text(t)
        out.append(n)
    return out


def appendix_8_utility_d(state: Dict[str, float], delta: Dict[str, float]) -> Dict[str, float]:
    out = dict(state)
    for k, v in delta.items():
        out[k] = float(max(0.0, min(1.0, out.get(k, 0.5) + v)))
    return out


def appendix_8_utility_e(samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    mode_counts: Dict[str, int] = {m: 0 for m in MODE_NAMES}
    cmd_counts: Dict[str, int] = {c: 0 for c in CMD_NAMES}

    for s in samples:
        mi = int(s.get("mode", 0))
        ci = int(s.get("command", 0))
        mode_counts[MODE_NAMES[mi % len(MODE_NAMES)]] += 1
        cmd_counts[CMD_NAMES[ci % len(CMD_NAMES)]] += 1

    return {
        "mode_counts": mode_counts,
        "cmd_counts": cmd_counts,
        "sample_count": len(samples),
    }


# smoke calls for appendix 8
_ = appendix_8_utility_a([0.1, 0.2, 0.4, 0.9])
_ = appendix_8_utility_b([0.1, 0.2, 0.4, 0.9])
_ = appendix_8_utility_c(["  Hello  World ", "මම ඔයාට ආදරෙයි"])
_ = appendix_8_utility_d({"dopamine": 0.5}, {"dopamine": 0.1})
_ = appendix_8_utility_e(train_records[:128])
